In [ ]:
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt, plotly.graph_objects as go
from ouysse import *
%autoreload 2

In [ ]:
BASE = r'Y:\MISSIONS\Eau\1 - Projet Hydrogéologique Ouysse\3 - Hydrodynamique\0 - Stations en continu\Alzou - Gramat\Gaetan'
CONSOLIDATED_PATH = os.path.join(BASE, 'Alzou_consolide.xlsx')
PUNCTUAL_PATH = os.path.join(BASE, 'Punctual measurements .xlsx')
OUTPUT_PATH = os.path.join(BASE, 'Alzou_interpole.xlsx')

NGF = None
MAX_HEURES = 12
SEUIL_IQR = 1.5

In [ ]:
df = pd.read_excel(CONSOLIDATED_PATH)
df['DATE'] = pd.to_datetime(df['DATE'], errors='coerce')
print(f"{len(df)} lignes, {df['DATE'].min()} à {df['DATE'].max()}")
appliquer_gammes(df, GAMMES)

In [ ]:
punctual = lire_points(PUNCTUAL_PATH, fuseau='Europe/Paris')
if len(punctual) > 0:
    print(f"{len(punctual)} points de contrôle")

In [ ]:
df_corrected = df.copy()
if len(punctual) > 0:
    for idx, point in punctual.iterrows():
        if point.get('Correction', 'Non').upper() == 'OUI':
            point_date = point['Datetime']
            for col in df_corrected.select_dtypes(include=[np.number]).columns:
                if 'Cond' in col and pd.notna(point.get('Conductivité')):
                    idx_nearest = (df_corrected['DATE'] - point_date).abs().idxmin()
                    mesure_value = df_corrected.loc[idx_nearest, col]
                    if pd.notna(mesure_value):
                        decalage = point['Conductivité'] - mesure_value
                        df_corrected.loc[df_corrected['DATE'] >= point_date, col] += decalage

In [ ]:
date_range = pd.date_range(df_corrected['DATE'].min(), df_corrected['DATE'].max(), freq='H')
df_grid = df_corrected.set_index('DATE').reindex(date_range).reset_index()
df_grid.rename(columns={'index': 'DATE'}, inplace=True)
colonnes_a_interpoler = df_grid.select_dtypes(include=[np.number]).columns.tolist()

In [ ]:
statuts = pd.DataFrame({'DATE': df_grid['DATE']})
for col in colonnes_a_interpoler:
    df_grid[f'Manquante_{col}'] = df_grid[col].isnull().astype(int)
    df_grid[f'Groupe_{col}'] = (df_grid[f'Manquante_{col}'] != df_grid[f'Manquante_{col}'].shift()).cumsum()
    groupe_sizes = df_grid.groupby(f'Groupe_{col}')[f'Manquante_{col}'].sum()
    trous_longs = groupe_sizes[groupe_sizes > MAX_HEURES].index
    df_grid[col] = df_grid[col].interpolate(method='linear', limit_direction='both')
    df_grid.loc[df_grid[f'Groupe_{col}'].isin(trous_longs), col] = np.nan
    statut_col = f'Statut_{col}'
    statuts[statut_col] = 'Brute'
    statuts.loc[df_grid[col].isnull(), statut_col] = 'Manquante'
    statuts.loc[(df_grid[col].notnull()) & (df_grid[f'Manquante_{col}'] == 1), statut_col] = 'Interpolée'
cols_to_drop = [c for c in df_grid.columns if c.startswith(('Manquante_', 'Groupe_'))]
df_grid = df_grid.drop(columns=cols_to_drop)
df_final = df_grid.merge(statuts, on='DATE', how='left')

In [ ]:
def ajouter_ngf(df, ngf_constant, colonne_niveau='Niveau_(cm)'):
    if colonne_niveau in df.columns and ngf_constant is not None:
        df['Niveau_(mNGF)'] = ngf_constant + df[colonne_niveau] / 100
    return df

# df_final = ajouter_ngf(df_final, ngf_constant=108.5)

In [ ]:
df_final.to_excel(OUTPUT_PATH, index=False)
print(f"Sauvegardé: {OUTPUT_PATH}")

In [ ]:
parametres_graph = [
    ('Niveau_(cm)', 'Niveau (cm)', 'lightseagreen'),
    ('Cond_Troll_(µS/cm)', 'Conductivité (µS/cm)', 'black'),
    ('température_(°C)', 'Température (°C)', 'crimson'),
    ('Turbidity_Troll_(NTU)', 'Turbidité (NTU)', 'darkorange'),
    ('O2_Troll_(mg/l)', 'Oxygène (mg/L)', 'darkmagenta'),
]

n = len(parametres_graph)
fig, axes = plt.subplots(n, 1, figsize=(14, 3*n), sharex=True)
if n == 1: axes = [axes]

for ax, (col, label, color) in zip(axes, parametres_graph):
    if col in df_final.columns:
        ax.plot(df_final['DATE'], df_final[col], color=color, linewidth=1, alpha=0.7)
        statut_col = f'Statut_{col}'
        if statut_col in df_final.columns:
            interp = df_final[df_final[statut_col] == 'Interpolée']
            if len(interp) > 0:
                ax.scatter(interp['DATE'], interp[col], s=5, color=color, alpha=0.3)
        ax.set_ylabel(label, fontsize=10)
        ax.grid(True, alpha=0.2)

axes[-1].set_xlabel('Date')
plt.tight_layout()
output_png = OUTPUT_PATH.replace('.xlsx', '_graphes.png')
plt.savefig(output_png, dpi=150, bbox_inches='tight')
print(f"Graphes: {output_png}")

In [ ]:
col_principal = 'Cond_Troll_(µS/cm)'
if col_principal in df_final.columns:
    fig_plotly = go.Figure()
    fig_plotly.add_trace(go.Scatter(
        x=df_final['DATE'], y=df_final[col_principal],
        mode='lines', name='Conductivité',
        line=dict(color='black', width=1.5)
    ))
    if len(punctual) > 0:
        fig_plotly.add_trace(go.Scatter(
            x=punctual['Datetime'], y=punctual['Conductivité'],
            mode='markers', name='Points contrôle',
            marker=dict(size=6, symbol='x', color='red')
        ))
    fig_plotly.update_layout(
        title='Alzou - Conductivité',
        xaxis_title='Date', yaxis_title='µS/cm',
        template='plotly_white', height=400
    )
    output_html = OUTPUT_PATH.replace('.xlsx', '_graphe_interactif.html')
    fig_plotly.write_html(output_html)